# spine-vision · Florence-2 fine-tune

Free Colab T4 is enough. Walk-through:

1. **Upload your dataset.** Drop `dataset.jsonl` and the matching `images/` folder into the Colab file pane (left sidebar → folder icon → drag the folder).
2. **Run all cells** (Runtime → Run all).
3. **Download the result** when the last cell finishes — it produces `model.onnx`.

## 1. Install dependencies

In [ ]:
!pip install -q transformers==4.45.0 accelerate==0.34.0 datasets==2.21.0 timm==1.0.9 einops==0.8.0 onnx==1.16.0 onnxruntime==1.19.0
!pip install -q Pillow

## 2. Load Florence-2-base

In [ ]:
import torch
from transformers import AutoProcessor, AutoModelForCausalLM

MODEL_ID = 'microsoft/Florence-2-base'
device = 'cuda' if torch.cuda.is_available() else 'cpu'
dtype = torch.float16 if device == 'cuda' else torch.float32

processor = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, trust_remote_code=True, torch_dtype=dtype).to(device)
print(f'Loaded {MODEL_ID} on {device}, dtype={dtype}')

## 3. Load the dataset

Expects `dataset.jsonl` (one row per image) and the matching `images/` directory in the working dir. Each row:

```json
{"image": "images/abc-123.jpg", "lines": ["The Bostonians", "Henry James"], ...}
```

In [ ]:
import json, random
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader

PROMPT_TASK = '<OCR_WITH_REGION>'  # Florence-2's OCR task token

class SpineDataset(Dataset):
    def __init__(self, jsonl_path):
        self.rows = [json.loads(l) for l in open(jsonl_path)]
    def __len__(self):
        return len(self.rows)
    def __getitem__(self, i):
        r = self.rows[i]
        img = Image.open(r['image']).convert('RGB')
        # Rotate 90° CCW: spines read vertically, Florence learns horizontally.
        img = img.rotate(90, expand=True)
        target_text = '\n'.join(r['lines'])
        return {'image': img, 'lines': r['lines'], 'target': target_text}

ds = SpineDataset('dataset.jsonl')
random.seed(42)
indices = list(range(len(ds)))
random.shuffle(indices)
split = max(1, int(len(ds) * 0.1))
val_idx = set(indices[:split])
print(f'{len(ds)} examples, {split} validation')

## 4. Train

Skeleton — replace with the canonical Florence-2 fine-tuning loop from the upstream notebook (https://github.com/roboflow/notebooks). The actual training is provider-specific enough that copying the official recipe is more reliable than rolling our own.

In [ ]:
# TODO: paste in the canonical Florence-2 fine-tuning loop and adapt:
#  - feed (image, target_text) where target_text is one line per book title
#  - use the <OCR> task prefix (not <OCR_WITH_REGION>) for v0; we add
#    bounding boxes in v1 when we also start segmenting individual spines
#  - val loss for early stopping; max 30 epochs; LR ~ 1e-6
#  - save final weights to ./spine-vision-ft/
raise NotImplementedError(
    'Replace this cell with the canonical Florence-2 fine-tuning loop. '
    'See https://github.com/roboflow/notebooks for an up-to-date recipe.'
)

## 5. Quick eval (sanity check)

Before exporting, eyeball that the model actually learned something. Real benchmarking happens in `eval/score.js`.

In [ ]:
from PIL import Image
import random

model.eval()
for i in random.sample(list(val_idx), 3):
    item = ds[i]
    inputs = processor(text='<OCR>', images=item['image'], return_tensors='pt').to(device, dtype)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=128, num_beams=3)
    pred = processor.batch_decode(out, skip_special_tokens=False)[0]
    print('GT :', item['target'])
    print('OUT:', pred)
    print('---')

## 6. Export to ONNX

ONNX is the format the browser adapter loads via `onnxruntime-web`. Quantize to int8 to keep the file under 200 MB.

In [ ]:
# TODO: ONNX export. Florence-2 has multiple sub-models (vision encoder,
# language decoder); both need exporting separately and the runner glues
# them. See the upstream Florence-2 ONNX export script.
#
# Target: ./model.onnx (or model_q.onnx after int8 quantization)
raise NotImplementedError('Replace with Florence-2 ONNX export script.')

## 7. Download the result

Drop `model.onnx` into the parent project's `public/spine-vision/` directory so the browser adapter can load it.

In [ ]:
from google.colab import files
files.download('model.onnx')